# 応用編14
V3.8新機能:代理ウォレット機能によるクロスデバイスログイン

このノートブックでは、スマートコントラクト・バージョン3.8から利用できる代理ウォレット機能について、使い方を説明します。

## 代理ウォレット機能の概要

代理ウォレット機能は、あるウォレット（元ウォレット）の権限を別のウォレット（代理ウォレット）に一時的・限定的に委任できる新機能です。主な利用シーンは、スマホ上のウォレットでPC上のアプリにログインするケース（クロスデバイス利用）や、元ウォレットの一部権限のみを代理ウォレットに委任するケースが想定されます。委任にあたって有効期限やアクセス権限の範囲を設定でき、必要に応じて一度限り有効といったモード指定も可能です。

## 本サンプルで想定するシナリオ

スマートフォン上のメインウォレットを使用し、PC上のブロックチェーンアプリにログインするケースです。ユーザーはスマホで保有するウォレット（元ウォレット）から、PCのブラウザアプリ内で一時生成した使い捨てウォレット（代理ウォレット）に権限を委任します。これにより、代理ウォレットが一定時間ユーザー本人としてトランザクションを発行できるようになり、トランザクションのたびにスマホ側で署名承認する手間が省けます（WalletConnectのような都度署名による遅延や失敗を回避できます）。

## 準備

設定やライブラリを読み込みます。  

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

adminWalletを元ウォレットとします。元ウォレットはスマホに保存されている想定です。  
元ウォレットのユーザIDを確認しておきます。

In [2]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'search', key: 'me' });
console.log(resp.value[0].id);

u86311649


動作確認用のスマートコントラクトをデプロイします。

In [3]:
var cid = await deploySmartContract({ a: 'number'}, function adv14(){
   return getCallerId();
});

## 代理ウォレットの作成

**想定：PC側**  
代理ウォレットを作成します。代理ウォレットはPCのブラウザアプリ内に保存される想定です。  
作成方法は、通常のウォレットと同じ方法で作成できます。

In [4]:
var proxyWallet = await api.importSigningWallet('es', await api.generateWalletKey('es'));

proxyWalletに対応するユーザIDがanonymousであることを確認しておきます。

In [5]:
var resp = await rpc.call(proxyWallet, 'c1query', { type: 'search', key: 'me' });
console.log(resp.value[0].id);

anonymous


proxyWalletのアドレスを表示します。  
実際のクロスデバイス環境では、このアドレスを含むQRコードがブラウザアプリの画面に表示する想定です。

In [6]:
var proxyAddress = proxyWallet.address;
console.log(proxyAddress);

ewbmk6bzkvENgEDNiU7GzHj9Xd4WDs


## 代理ウォレットへの委任

**想定：スマホ側**  
元ウォレットから代理ウォレットへ委任します。  
委任の状態はブロックチェーンに記録されます。

In [7]:
var resp = await rpc.call(adminWallet, 'c1walletproxy', {
    addr: proxyAddress, // 実際のクロスデバイス環境では、スマホのカメラ機能を使って、代理ウォレットのアドレスをQRコードから読み取る想定
    expiry: Date.now() + 3600*1000, // 委任の有効期限（現在時刻＋1時間）
    access: { write_tx: 'unlimited', query: 'unlimited' }, // 書き込みトランザクションの発行とクエリ呼び出しを無制限に委託
});
console.log(resp);

{
  txno: 40288,
  txid: 'xEN8oQRSdAu47Ki2CsgyPofiPTt6pnp5gMZ7Zs8EkArUr',
  status: 'ok',
  value: null
}


## 代理ウォレットが有効になったことの確認

**想定：PC側**  
代理ウォレットが有効になるまで待ちます。  
ブロックチェーンに代理ウォレットの状態を問い合わせることにより、確認します。  
このループは、c1queryで代理ウォレットが有効になるまで繰り返し問い合わせています。  
これは、クロスデバイス環境ではスマホ側の操作が完了したことが直接にはわからないため。

In [8]:
while (true) {
    await new Promise(resolve => setTimeout(resolve, 1000)); // 1秒まつ
    var resp = await rpc.call(proxyWallet, 'c1query', { type: 'a_proxy' }); // 代理ウォレットの状態を取得
    if (resp.status === 'ok') break; // 代理ウォレットであればループから抜ける
}
console.log(resp.value); // 代理ウォレットの状態を表示

{
  addr: 'ewbmk6bzkvENgEDNiU7GzHj9Xd4WDs',
  origin: 'eSQgkupyuMPkx22SuEui6HjUHyvTqm',
  uid: 'u86311649',
  expiry: 1763548783824,
  access: { write_tx: 'unlimited', query: 'unlimited' },
  mode: []
}


addrは代理ウォレットのアドレス、originは元ウォレットのアドレス、uidは元ウォレットのユーザID、となっています。  
expiryとaccessは、代理ウォレットを委任した時の設定となっています。

## 代理ウォレットを使ったブロックチェーンへのアクセス

**想定：PC側**  
代理ウォレットを使ったアクセスは、元ウォレットを使ったアクセスと同じように動作します。

proxyWalletのユーザIDを問い合わせると、元ウォレットのユーザIDが返ります。  

In [9]:
var resp = await rpc.call(proxyWallet, 'c1query', { type: 'search', key: 'me' });
console.log(resp.value[0].id);

u86311649


トランザクションを呼び出すことができ、そのときのcallerは元ウォレットのユーザIDとなります。

In [10]:
var resp = await rpc.call(proxyWallet, cid);
console.log(resp);
var txno = resp.txno;

{
  txno: 40289,
  txid: 'xUad9sAoLEhEJmiVJ8si7w262JynK5bkQCJzi4mXiLnFy',
  status: 'ok',
  value: 'u86311649'
}


トランザクションの履歴を見ても、callerは元ウォレットのユーザIDとなります。  
ただし、トランザクションのウォレットアドレスは代理ウォレットのものとなります。

In [11]:
var resp = await rpc.call(proxyWallet, 'c1query', { type: 'a_transaction', id: txno });
console.log(resp.value);

{
  blkno: 28505,
  time: 1763545186556,
  txid: 'xUad9sAoLEhEJmiVJ8si7w262JynK5bkQCJzi4mXiLnFy',
  addr: 'ewbmk6bzkvENgEDNiU7GzHj9Xd4WDs',
  caller: [ 'u86311649', 'admin@sample_codes' ],
  callee: [ 'c039384934', 'adv14@sample_codes' ],
  status: 'ok',
  pack64: 'BQAAAAEAAADOAAAAAgAAAEAAAABAAAAAA3siY29udHJhY3QiOiJjMDM5Mzg0OTM0IiwiYXJncyI6e30sImFkZHIiOiJld2JtazZiemt2RU5nRUROaVU3R3pIajlYZDRXRHMiLCJkZWFkbGluZSI6MTc2MzU0NTI4NjM2MSwib3JhY2xlIjoiU2I0S0xTQ3FOcWtBY2EyZCIsImJsb2NrcmVmIjp7Im5vIjoyODUwNCwiaGFzaCI6InovR3FiUmpwSUFIV1JJL1dsMWUxN2ZtZ0xyUUtEWFhyV0U0dXFLUEVIMnM9In19ZXNf2Esvp9V/+YZf9YoLSV1KnLtljM+q3npNwxYg2RsmGEI9zqxxha3dI5RA9R22ujopCVPLf5RiADO8ogMFhfuxtb+e7gmI0qGfk39uyLNufAwSgMj1kpK2EG3QiU96CoQcLll+LEIJAqy4y9ZW5ER3LWyVijWAtEi6PtA3Dn8Lgg==',
  txno: 40289,
  caller_txno: 0,
  argstr: '{}',
  valuestr: '"u86311649"',
  subtxs: 0,
  steps: 205,
  disclosed_to: [],
  related_to: [],
  cur_disclosed_to: [
    [ 'u86311649', 'admin@sample_codes' ],
    [ 'c039384934', 'adv14@sample_codes' 

## 代理ウォレットの解除

**想定：スマホ側**  
元ウォレットから代理ウォレットへの委任を途中で解除できます。  
具体的方法は、expiryを0に設定して再委任を行います。

In [12]:
var resp = await rpc.call(adminWallet, 'c1walletproxy', {addr: proxyAddress, expiry: 0, access: {} }); 
console.log(resp);

{
  txno: 40290,
  txid: 'xDsci4oVuxkGxkJczUELMkhMvKx42wY2bKhHfHo3Z6ZLr',
  status: 'ok',
  value: null
}


## 代理ウォレットが解除された後の動作

**想定：PC側**  
代理ウォレットが解除された後は、通常のanonymousウォレットとして動作します。

proxyWalletのユーザIDを問い合わせると、anonymousが返ります。  

In [13]:
var resp = await rpc.call(proxyWallet, 'c1query', { type: 'search', key: 'me' });
console.log(resp.value[0].id);

anonymous


コントラクトの呼び出しもanonymousユーザでの呼び出しとなり、この場合は実行権限がないためエラーになります。

In [14]:
var resp = await rpc.call(proxyWallet, cid);
console.log(resp);

{
  txno: 40291,
  txid: 'xvogdpY2zYE7XdqaqdvRxJ4oyeQfFoavy3HUffbNF3p3NB',
  status: 'denied',
  value: 'accessible permission'
}
